# 01 — Exploratory Data Analysis
Egyptian ID OCR Dataset

In [ ]:
import sys
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
from pathlib import Path
from src.data.dataset import EgyptianIDDataset, FIELD_CLASSES

DATASET_ROOT = '../Egyptain-Person-ID-1'

## Dataset Statistics

In [ ]:
for split in ['train', 'valid', 'test']:
    ds = EgyptianIDDataset(DATASET_ROOT, split=split)
    stats = ds.get_split_stats()
    print(f"\n{'='*40}")
    print(f"Split: {stats['split']}")
    print(f"  Images: {stats['total_images']}")
    print(f"  Annotations: {stats['total_annotations']}")

## Class Distribution

In [ ]:
train_ds = EgyptianIDDataset(DATASET_ROOT, split='train')
dist = train_ds.get_class_distribution()

fig, ax = plt.subplots(figsize=(14, 5))
classes = list(dist.keys())
counts = list(dist.values())
bars = ax.barh(classes, counts, color='steelblue')
ax.set_xlabel('Annotation Count')
ax.set_title('Class Distribution — Training Set')
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Sample Images with Annotations

In [ ]:
from matplotlib.patches import Rectangle
import random
from src.data.dataset import IDX_TO_CLASS

COLORS = plt.cm.tab20.colors

def draw_sample(idx, ds):
    item = ds[idx]
    img = item['image']
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    
    for box, cls_id in zip(item['boxes'], item['class_ids']):
        cx, cy, bw, bh = box
        x1 = (cx - bw/2) * w
        y1 = (cy - bh/2) * h
        bw_px = bw * w
        bh_px = bh * h
        color = COLORS[cls_id % len(COLORS)]
        rect = Rectangle((x1, y1), bw_px, bh_px, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, IDX_TO_CLASS[cls_id], color=color, fontsize=8, fontweight='bold')
    
    ax.set_title(f'Sample {idx}: {Path(item["image_path"]).name}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

for i in random.sample(range(len(train_ds)), min(3, len(train_ds))):
    draw_sample(i, train_ds)

## Image Size Distribution

In [ ]:
from tqdm import tqdm

widths, heights = [], []
img_dir = Path(DATASET_ROOT) / 'train' / 'images'
for p in tqdm(list(img_dir.glob('*.jpg'))[:200]):
    img = cv2.imread(str(p))
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color='steelblue')
axes[0].set_title('Width Distribution')
axes[1].hist(heights, bins=20, color='coral')
axes[1].set_title('Height Distribution')
plt.suptitle('Image Dimensions (sample of 200 training images)')
plt.tight_layout()
plt.show()
print(f'Width  — mean={np.mean(widths):.0f}, std={np.std(widths):.0f}')
print(f'Height — mean={np.mean(heights):.0f}, std={np.std(heights):.0f}')